In [19]:
%pip install -U -q "google-genai>=1.0.0" "pypdf"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google import genai
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [22]:
from pypdf import PdfReader

# Function to extract text from PDF
def extract_pdf_text(pdf_path):
    """Extract all text from a PDF file"""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text.strip()

# Load your CV and Profile files
cv_text = extract_pdf_text("New_Resume.pdf")
profile_text = extract_pdf_text("Profile.pdf")

print(f"CV extracted: {len(cv_text)} characters")
print(f"Profile extracted: {len(profile_text)} characters")


CV extracted: 3445 characters
Profile extracted: 3679 characters


In [23]:
from google.genai import types
title = "My Portfolio RAG Voice Assistant"
sample_text = """

"""

EMBEDDING_MODEL_ID = MODEL_ID = "gemini-embedding-001"
embedding = client.models.embed_content(
    model = EMBEDDING_MODEL_ID,
    contents=sample_text,
    config=types.EmbedContentConfig(
        task_type="retrieval_document",
        title=title
    )
)

print(embedding)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) embeddings=[ContentEmbedding(
  values=[
    0.0003545584,
    0.014725423,
    0.006064026,
    -0.07896516,
    0.0065108053,
    <... 3067 more items ...>,
  ]
)] metadata=None


In [26]:
Document1 = {
    "title": "My Resume",
    "text": cv_text
}

Document2 = {
    "title": "My Profile",
    "text": profile_text
}

documents = [Document1, Document2]

# Embed all documents and store embeddings
embeddings_list = []
for i, doc in enumerate(documents):
    embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=doc["text"],
        config=types.EmbedContentConfig(
            task_type="retrieval_document",
            title=doc["title"]
        )
    )
    doc["embedding"] = embedding.embeddings[0].values
    embeddings_list.append(embedding.embeddings[0].values)
    print(f"\n✓ {doc['title']} embedded successfully")
    print(f"  Content length: {len(doc['text'])} characters")
    print(f"  Embedding dimension: {len(embedding.embeddings[0].values)}")



✓ My Resume embedded successfully
  Content length: 3445 characters
  Embedding dimension: 3072

✓ My Profile embedded successfully
  Content length: 3679 characters
  Embedding dimension: 3072


In [40]:
import pandas as pd

df = pd.DataFrame({
    'Title': [doc['title'] for doc in documents],
    'Text': [doc['text'] for doc in documents],
    'Embeddings': [doc['embedding'] for doc in documents]
})
print(df)


        Title                                               Text  \
0   My Resume  Clency Christine\nApplying for: OpenAI Researc...   
1  My Profile  Contact\nclency2023@gmail.com\nwww.linkedin.co...   

                                          Embeddings  
0  [-0.014870569, 0.023160255, 0.004667831, -0.04...  
1  [-0.015462844, 0.013898114, -0.0036821149, -0....  


In [31]:
query = "Where does clency go to school?"
request = client.models.embed_content(
    model = EMBEDDING_MODEL_ID,
    contents=query,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
    )
)

print(request)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) embeddings=[ContentEmbedding(
  values=[
    -0.030241983,
    0.011863463,
    0.012215773,
    -0.05494895,
    -0.011369359,
    <... 3067 more items ...>,
  ]
)] metadata=None


In [34]:
import numpy as np

def find_best_passage(query, dataframe):
  """
  Compute the distances between the query and each document in the dataframe
  using the dot product.
  """
  query_embedding = client.models.embed_content(
      model=EMBEDDING_MODEL_ID,
      contents=query,
      config=types.EmbedContentConfig(
          task_type="retrieval_document",
          )
  )

  dot_products = np.dot(
      np.stack(dataframe['Embeddings']),
      query_embedding.embeddings[0].values
  )
  idx = np.argmax(dot_products)
  return dataframe.iloc[idx]['Text']

In [35]:
from IPython.display import Markdown

passage = find_best_passage(query, df)
print("Query:", query)
print("\nRetrieved Answer:")
print(passage)
print("\n" + "="*50)
Markdown(passage)


Query: Where does clency go to school?

Retrieved Answer:
Contact
clency2023@gmail.com
www.linkedin.com/in/clency-
christine-643b32265 (LinkedIn)
Top Skills
Working under Pressure
Project Planning
Pitching
Certifications
Building RAG Apps Using MongoDB
GirlCode 2025 Participant and 2nd
Position
Taifa Teule Leadership Network
Hackathon Participant
ALX Software Engineering
Certificate
Clency Christine
Lifelong Builder | Backend Developer - |Python|Django rest
framework|Fastapi|Rest APIs, Android Developer - Kotlin, Jetpack
Compose, Learning AI integration into my apps(Applied AI)
Nairobi County, Kenya
Summary
I'm obsessed with understanding intelligence how it emerges, how
it learns, and how we can build systems that push the boundaries
of what's possible. OpenAI's mission to ensure artificial general
intelligence benefits all of humanity deeply resonates with me; I
believe the most transformative breakthroughs will come from bold,
principled exploration at the frontier.
My path into AI 

Contact
clency2023@gmail.com
www.linkedin.com/in/clency-
christine-643b32265 (LinkedIn)
Top Skills
Working under Pressure
Project Planning
Pitching
Certifications
Building RAG Apps Using MongoDB
GirlCode 2025 Participant and 2nd
Position
Taifa Teule Leadership Network
Hackathon Participant
ALX Software Engineering
Certificate
Clency Christine
Lifelong Builder | Backend Developer - |Python|Django rest
framework|Fastapi|Rest APIs, Android Developer - Kotlin, Jetpack
Compose, Learning AI integration into my apps(Applied AI)
Nairobi County, Kenya
Summary
I'm obsessed with understanding intelligence how it emerges, how
it learns, and how we can build systems that push the boundaries
of what's possible. OpenAI's mission to ensure artificial general
intelligence benefits all of humanity deeply resonates with me; I
believe the most transformative breakthroughs will come from bold,
principled exploration at the frontier.
My path into AI has been anything but conventional. With a
foundation in software engineering and mathematics, I taught myself
machine learning through relentless self-study—diving into PyTorch
tutorials, implementing papers from scratch, and experimenting late
into the night. What fuels me most is reading cutting-edge research:
I regularly lose myself in MIT CSAIL papers, ArXiv preprints from
leading minds like Ilya Sutskever, Yann LeCun, and Karol Hausner,
and deep dives into works from DeepMind, Anthropic, and OpenAI.
There's nothing more thrilling than dissecting a new architecture,
understanding its theoretical underpinnings, and immediately
prototyping it.
I've turned that curiosity into action through ambitious projects. As
part of an all-women team, I co-built GKash, an AI-powered mobile
app teaching saving and investing to underserved communities
in Kenya—earning 2nd place in the 2025 Absa GirlCodeHack (a
Pan-African women-in-tech competition) and acceptance into a
competitive accelerator. I've developed backend systems for social
impact platforms (e.g., GBV detection in workplaces using Slack
integrations), built real-time fraud detection models, and contributed
to community-driven tools like Techi-Pro Konnect and ProcureGuard
AI. These experiences honed my ability to rapidly prototype, iterate,
and ship complex systems independently.
  Page 1 of 3
   
I'm extremely comfortable with advanced mathematics—linear
algebra, probability, statistics, and calculus—and proficient in
Python.
Experience
GDG On Campus JKUAT
Google Developer Group on Campus organizer JKUAT
September 2025 - Present (5 months)
Organizing events, sessions and partnerships for the gdg developer
community at JKUAT.
Tech For Nonprofits
Django Backend Developer (Volunteer)
June 2025 - Present (8 months)
Build and maintain scalable REST APIs using Django & Django REST
Framework
– Design and optimize database models and queries (PostgreSQL)
– Implement authentication, authorization, and security best practices
– Integrate backend with frontend systems and third-party services
– Write clean, testable, production-ready code; participate in code reviews
– Collaborate with frontend devs, product managers, and stakeholders for
feature delivery
TechiPro Konnect App
Co-Founder
June 2024 - Present (1 year 8 months)
Nairobi County, Kenya
This is an android developer role.
HakiChain
Software Developer
June 2025 - November 2025 (6 months)
Education
Jomo Kenyatta University of Agriculture and Technology (JKUAT)
Bachelor's degree, Mathematics and Computer Science · (2023 - 2027)
  Page 2 of 3
   
ALX software engineering programme
Software Engineering · (2023 - 2024)
eMobilis Mobile Technology Institute
 · (January 2023 - May 2023)
  Page 3 of 3

In [36]:
import textwrap

def make_prompt(query, relevant_passage):
  escaped = (
      relevant_passage
      .replace("'", "")
      .replace('"', "")
      .replace("\n", " ")
  )
  prompt = textwrap.dedent("""
    You are a helpful and informative bot that answers questions using text
    from the reference passage included below. Be sure to respond in a
    complete sentence, being comprehensive, including all relevant
    background information.

    However, you are talking to a non-technical audience, so be sure to
    break down complicated concepts and strike a friendly and conversational
    tone. If the passage is irrelevant to the answer, you may ignore it.

    QUESTION: '{query}'
    PASSAGE: '{relevant_passage}'

    ANSWER:
  """).format(query=query, relevant_passage=escaped)


  return prompt

In [37]:
prompt = make_prompt(query, passage)
Markdown(prompt)




You are a helpful and informative bot that answers questions using text
from the reference passage included below. Be sure to respond in a
complete sentence, being comprehensive, including all relevant
background information.

However, you are talking to a non-technical audience, so be sure to
break down complicated concepts and strike a friendly and conversational
tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: 'Where does clency go to school?'
PASSAGE: 'Contact clency2023@gmail.com www.linkedin.com/in/clency- christine-643b32265 (LinkedIn) Top Skills Working under Pressure Project Planning Pitching Certifications Building RAG Apps Using MongoDB GirlCode 2025 Participant and 2nd Position Taifa Teule Leadership Network Hackathon Participant ALX Software Engineering Certificate Clency Christine Lifelong Builder | Backend Developer - |Python|Django rest framework|Fastapi|Rest APIs, Android Developer - Kotlin, Jetpack Compose, Learning AI integration into my apps(Applied AI) Nairobi County, Kenya Summary Im obsessed with understanding intelligence how it emerges, how it learns, and how we can build systems that push the boundaries of whats possible. OpenAIs mission to ensure artificial general intelligence benefits all of humanity deeply resonates with me; I believe the most transformative breakthroughs will come from bold, principled exploration at the frontier. My path into AI has been anything but conventional. With a foundation in software engineering and mathematics, I taught myself machine learning through relentless self-study—diving into PyTorch tutorials, implementing papers from scratch, and experimenting late into the night. What fuels me most is reading cutting-edge research: I regularly lose myself in MIT CSAIL papers, ArXiv preprints from leading minds like Ilya Sutskever, Yann LeCun, and Karol Hausner, and deep dives into works from DeepMind, Anthropic, and OpenAI. Theres nothing more thrilling than dissecting a new architecture, understanding its theoretical underpinnings, and immediately prototyping it. Ive turned that curiosity into action through ambitious projects. As part of an all-women team, I co-built GKash, an AI-powered mobile app teaching saving and investing to underserved communities in Kenya—earning 2nd place in the 2025 Absa GirlCodeHack (a Pan-African women-in-tech competition) and acceptance into a competitive accelerator. Ive developed backend systems for social impact platforms (e.g., GBV detection in workplaces using Slack integrations), built real-time fraud detection models, and contributed to community-driven tools like Techi-Pro Konnect and ProcureGuard AI. These experiences honed my ability to rapidly prototype, iterate, and ship complex systems independently.   Page 1 of 3     Im extremely comfortable with advanced mathematics—linear algebra, probability, statistics, and calculus—and proficient in Python. Experience GDG On Campus JKUAT Google Developer Group on Campus organizer JKUAT September 2025 - Present (5 months) Organizing events, sessions and partnerships for the gdg developer community at JKUAT. Tech For Nonprofits Django Backend Developer (Volunteer) June 2025 - Present (8 months) Build and maintain scalable REST APIs using Django & Django REST Framework – Design and optimize database models and queries (PostgreSQL) – Implement authentication, authorization, and security best practices – Integrate backend with frontend systems and third-party services – Write clean, testable, production-ready code; participate in code reviews – Collaborate with frontend devs, product managers, and stakeholders for feature delivery TechiPro Konnect App Co-Founder June 2024 - Present (1 year 8 months) Nairobi County, Kenya This is an android developer role. HakiChain Software Developer June 2025 - November 2025 (6 months) Education Jomo Kenyatta University of Agriculture and Technology (JKUAT) Bachelors degree, Mathematics and Computer Science · (2023 - 2027)   Page 2 of 3     ALX software engineering programme Software Engineering · (2023 - 2024) eMobilis Mobile Technology Institute  · (January 2023 - May 2023)   Page 3 of 3'

ANSWER:


In [38]:
MODEL_ID = "gemini-2.5-flash"
answer = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)

In [39]:
Markdown(answer.text)

It looks like Clency has a strong educational background! She is currently pursuing a Bachelor's degree in Mathematics and Computer Science at **Jomo Kenyatta University of Agriculture and Technology (JKUAT)**, with her studies expected to run from 2023 to 2027. She also completed an ALX software engineering programme from 2023 to 2024 and attended eMobilis Mobile Technology Institute from January to May 2023.